### VectorStore

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance
from sentence_transformers import SentenceTransformer

 
model = SentenceTransformer("BAAI/bge-large-en")

 
client = QdrantClient(url="http://localhost:6333")

 
collection_name = "gitwise_code"

# Create / recreate collection
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

# chunks
chunk1 = "How does authentication work?"
chunk2 = "How to validate token?"

# Generating embeddings
emb_chunk1 = model.encode(chunk1, normalize_embeddings=True)
emb_chunk2 = model.encode(chunk2, normalize_embeddings=True)

# Convert embeddings to list
emb_chunk1 = emb_chunk1.tolist()
emb_chunk2 = emb_chunk2.tolist()

# if want to create vector for insertion without id then use plain dictionary else use PointStruct
points = [
    PointStruct(
        id=1,
        vector=emb_chunk1,
        payload={
            "text": chunk1,
            "file": "auth_service.py",
            "function": "login_user",
        },
    ),
    PointStruct(
        id=2,
        vector=emb_chunk2,
        payload={
            "text": chunk2,
            "file": "jwt_utils.py",
            "function": "validate_token",
        },
    ),
]

#update if available insert if not
client.upsert(collection_name=collection_name, points=points)

# Creating query embedding
query_vector = model.encode(
    "How does authentication work?",
    normalize_embeddings=True
).tolist()

 
results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=5,
    with_payload=True,
    with_vectors=True
)
 
for point in results.points:
    print("Score:", point.score)
    print("Payload:", point.payload)
    print("Vector:", point.vector)
    print()

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3305.97it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\HP\AppData\Local\Temp\ipykernel_3340\3852206357.py:15: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Score: 1.0000001
Payload: {'text': 'How does authentication work?', 'file': 'auth_service.py', 'function': 'login_user'}
Vector: [0.0018811785, -0.0044286563, -0.011022817, -0.005623735, 0.002267355, 0.000167347, 0.010106856, 0.03760032, 0.0079144165, 0.05127985, 0.019758843, 0.037365347, 0.008124557, -0.010328202, -0.03742793, 0.030739335, -0.0041870913, -6.4115346e-05, -0.024056597, 0.01217137, 0.019441333, -0.017303765, -0.0643304, -0.010727573, -0.009521078, 0.013347712, 0.03971055, 0.0026490872, 0.051985037, 0.06195185, -0.03402581, 0.021763546, 0.011192091, -0.046132706, -0.045019403, -0.04716843, 0.004284311, -0.058497876, -0.04701291, -0.003959425, -0.005226243, 0.012525034, 0.016187163, -0.0450081, -0.0818951, 0.0069865296, -0.03945485, -0.038033325, 0.003989768, -0.02483096, 0.011814173, 0.010880083, 0.012147446, 0.021667698, -0.0065739034, 0.01218132, -0.012911576, 0.026941616, -0.0257443, 0.02442642, 0.012025667, 0.03706679, 0.017874744, -0.03394163, 0.015677426, 0.01442184